In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from sentence_transformers import SentenceTransformer

from src.data import load_dataset
from src.embedder import record_to_text, get_embeddings, save_embeddings, load_embeddings, get_cache_path
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import MODELS, TOP_K

corpus, questions, categories = load_dataset()
corpus_texts = [record_to_text(item) for item in corpus]

print(f"Корпус: {len(corpus)} фрагментов")
print(f"Вопросы: {len(questions)} вопросов")

Корпус: 200 фрагментов
Вопросы: 25 вопросов


In [2]:
model_name = MODELS[0]
cache_path = get_cache_path(model_name)

if Path(cache_path).exists():
    corpus_embedding = load_embeddings(cache_path)
    print("Загружена из кэша")
else:
    corpus_embedding = get_embeddings(corpus_texts, model_name)
    save_embeddings(cache_path, corpus_embedding)
    print("Вычислено и добавлено в кэш")

Загружена из кэша


In [3]:
model = SentenceTransformer(model_name)
search_outputs = search_all_questions(questions, model, corpus, corpus_embedding)

metrics = evaluate(search_outputs, k=TOP_K)
print(metrics)
print("Для модели:", model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'precision_at_3': 0.84, 'mrr': 0.56, 'num_questions': 25}
Для модели: paraphrase-multilingual-MiniLM-L12-v2


In [4]:
retrieved_ids_list = []
for search_result in search_outputs:
    top_ids = [result["id"] for result in search_result["results"][:TOP_K]]
    retrieved_ids_list.append(top_ids)

In [5]:
errors = []
corpus_dict = {item["id"]: item for item in corpus}

for i, (q, retrieved_ids) in enumerate(zip(questions, retrieved_ids_list)):
    correct_id = q["correct_chunk_id"]
    if correct_id not in retrieved_ids:
        correct_func = corpus_dict.get(correct_id, {})
        wrong_funcs = [corpus_dict.get(rid, {}) for rid in retrieved_ids]
        
        errors.append({
            "question_id": q["question_id"],
            "query": q["query"],
            "query_lang": q.get("language", "unknown"),
            "correct_func_name": correct_func.get("function_name", "unknown"),
            "correct_category": correct_func.get("category", "unknown"),
            "retrieved_names": [f.get("function_name", "unknown") for f in wrong_funcs],
            "retrieved_categories": [f.get("category", "unknown") for f in wrong_funcs],
        })

print(f"\nВсего вопросов: {len(questions)}")
print(f"Ошибок: {len(errors)}")

if errors:
    print("\n" + "="*80)
    print("ТАБЛИЦА ОШИБОК")
    print("="*80)
    
    for err in errors:
        print(f"\n Вопрос {err['question_id']}: {err['query']}")
        print(f"   Должно быть: {err['correct_func_name']} ({err['correct_category']})")
        print(f"   Нашлось: {err['retrieved_names']} ({err['retrieved_categories']})")


Всего вопросов: 25
Ошибок: 4

ТАБЛИЦА ОШИБОК

 Вопрос q_01: как проверить, истёк ли токен?
   Должно быть: verify_jwt_token (auth)
   Нашлось: ['validate_inn', 'check_password', 'validate_snils'] (['validation', 'auth', 'validation'])

 Вопрос q_08: массовая вставка большого количества записей
   Должно быть: bulkInsertRecords (database)
   Нашлось: ['countUsers', 'bulk_insert_records', 'format_bytes'] (['database', 'database', 'utils'])

 Вопрос q_11: обработка ситуации, когда сервер вернул 429
   Должно быть: handle_rate_limit (http)
   Нашлось: ['handleRateLimit', 'restoreFromBackup', 'rollbackTransaction'] (['http', 'database', 'database'])

 Вопрос q_24: compute file hash for verification
   Должно быть: calculateHash (utils)
   Нашлось: ['calculate_hash', 'validateCredentials', 'validateInn'] (['utils', 'auth', 'validation'])


In [6]:
import json
from pathlib import Path

outputs_dir = Path.cwd().parent / "outputs"
outputs_dir.mkdir(exist_ok=True)

output_path = outputs_dir / "errors_minilm.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\nОшибки MiniLM сохранены в {output_path}")


Ошибки MiniLM сохранены в C:\CPP\semantic-search-case\outputs\errors_minilm.json
